In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr import Unit, Configuration, Variable

from utils.setup_utils import DEV_LOCATION_ID_LIST

### Setup

In [ ]:
spark = create_spark_session()

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
# configure download
ev.download.configure(
    api_key='your_key_here'
)

### Add domain data

#### Pull domain tables from teehr warehouse

In [ ]:
print("Loading domain tables.")

print("Loading configurations.")
ev.download.configurations(
    load=True
)

print("Loading units.")
ev.download.units(
    load=True
)

print("Loading variables.")
ev.download.variables(
    load=True
)

print("Loading locations.")
ev.download.locations(
    ids=DEV_LOCATION_ID_LIST,
    load=True
)

print("Loading location croswalks.")
ev.download.location_crosswalks(
    primary_location_id=DEV_LOCATION_ID_LIST,
    load=True
)

print("Loading location attributes.")
ev.download.attributes(
    load=True
)
ev.download.location_attributes(
    location_id=DEV_LOCATION_ID_LIST,
    load=True
)

print("Loading domain variables, locations, crosswalks, and attributes complete!")

#### Add general domain entries to support FVED data ingest

In [ ]:
# add unit for volume
unit = Unit(
    name='m^3',
    long_name='Cubic Meter'
)
ev.units.add(unit)

#### Add domain entries to support cbrfc_unregulated_esp_forecast configuration data

In [ ]:
# add configurations for cbrfc wsv's (use existing for usgs/nwm)
configuration = Configuration(
    name="cbrfc_unregulated_esp_forecast",
    timeseries_type="secondary",
    description="CBRFC unregulated ESP water supply volume forecasts",
)
ev.configurations.add(configuration)

# add variable for monthly wsv's
variable = Variable(
    name="volume_monthly_sum",
    long_name="Monthly summed water supply volume"
)
ev.variables.add(variable)

### Kill spark

In [ ]:
spark.stop()